# 00 — محرّك التحليل الببليومتري | Bibliometric starter

**الهدف Goal:** تحويل `literature/sources.csv` إلى جداول ورسوم جاهزة لقسم *Results*.

**قبل التشغيل Before running:** أضف 30+ مصدراً بـ `python tools/litkit.py add ...`

المخرجات | Outputs → `experiments/results/figures/` و`experiments/results/tables/`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'literature' / 'sources.csv'
OUT = ROOT / 'experiments' / 'results'
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
(OUT / 'tables').mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SRC)
print('all sources:', len(df))
df = df[df['include'].str.lower() == 'yes']
print('included   :', len(df))
df.head(3)

## 1) التوزيع الزمني | Year distribution

In [ ]:
by_year = df['year'].value_counts().sort_index()
ax = by_year.plot(kind='bar', figsize=(6, 3.4), color='#2b6cb0', title='Included sources by year')
ax.set_ylabel('Sources')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'fig1_year_distribution.png', dpi=300)
by_year.to_csv(OUT / 'tables' / 'tab_year.csv')
by_year

## 2) نوع المنشور وتصنيفه | Type & rank

In [ ]:
print(df['type'].value_counts())
print()
print(df['venue_rank'].value_counts())

## 3) أكواد المشاكل والحلول | Problem & solution codes

نفكّك الأعمدة المفصولة بفاصلة إلى صفوف (explode) للعدّ.

In [ ]:
def explode(col):
    s = (df[col].fillna('').str.upper()
           .str.replace(' ', '', regex=False)
           .str.split(',')
           .explode())
    return s[(s != '') & (s != 'NA')]

problems = explode('problem_codes')
solutions = explode('solution_codes')

top_problems = problems.value_counts()
ax = top_problems.head(12).iloc[::-1].plot(
    kind='barh', figsize=(6, 4), color='#2f855a', title='Most frequently coded problems')
ax.set_xlabel('Sources')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'fig2_problem_frequency.png', dpi=300)
top_problems.to_csv(OUT / 'tables' / 'tab_problems.csv')
solutions.value_counts().to_csv(OUT / 'tables' / 'tab_solutions.csv')
top_problems

## 4) خريطة حرارية: مشكلة × حل 🔥 | Problem × solution heatmap

**هذه أهم نتيجة في الورقة**: تُظهر أين تركّزت الحلول (خلايا داكنة) وأين الفجوة (خلايا فارغة).

In [ ]:
rows = []
for _, r in df.iterrows():
    ps = [c.strip().upper() for c in str(r['problem_codes']).split(',') if c.strip() and c.strip().upper() != 'NA']
    ss = [c.strip().upper() for c in str(r['solution_codes']).split(',') if c.strip() and c.strip().upper() != 'NA']
    for p in ps:
        for s in ss:
            rows.append({'problem': p, 'solution': s})

pairs = pd.DataFrame(rows)
heat = pd.crosstab(pairs['problem'], pairs['solution'])
heat.to_csv(OUT / 'tables' / 'tab_problem_solution_matrix.csv')

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
cax = ax.imshow(heat.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns, rotation=90)
ax.set_yticks(range(len(heat.index)));   ax.set_yticklabels(heat.index)
ax.set_xlabel('Solution family (S#)'); ax.set_ylabel('Problem (T#/O#)')
ax.set_title('Problem x Solution coverage — empty cells = research gaps')
fig.colorbar(cax, label='Sources')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'fig3_problem_solution_heatmap.png', dpi=300)
heat

## 5) تصدير جداول جاهزة للورقة | Export tables for the manuscript

In [ ]:
summary = pd.DataFrame({
    'n_sources': [len(df)],
    'n_problem_codes': [problems.nunique()],
    'n_solution_codes': [solutions.nunique()],
    'year_min': [int(df['year'].min())] if len(df) else [None],
    'year_max': [int(df['year'].max())] if len(df) else [None],
    'pct_journal': [round((df['type'] == 'journal').mean() * 100, 1)],
})
summary.to_csv(OUT / 'tables' / 'tab_summary.csv', index=False)
print('tables written to', OUT / 'tables')
summary.T

## ✅ الخطوة التالية | Next step

1. انسخ الأرقام إلى `manuscript/04_results.md`.
2. اكتب الفجوة في `manuscript/05_discussion.md` بصيغة:
   *"Only N of the M included studies (X%) addressed problem P with solution S."*
3. شغّل `python tools/validate_sources.py` قبل كل تصدير نهائي.